In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# 读取数据
df = pd.read_excel(r"data\全部中餐厅数据.xlsx")

# 清理
df = df.dropna(subset=["price_level", "stars", "review_count", "city"])

# 对数评论数
df["log_review"] = np.log(df["review_count"])

# ====================== 构造工具变量 ======================
# 同一城市价格中位数
df["iv_price"] = df.groupby("city")["price_level"].transform("median")

# ====================== OLS ======================
print("\n===== OLS =====")
X_ols = sm.add_constant(df[["price_level", "log_review"]])
y = df["stars"]

ols_model = sm.OLS(y, X_ols).fit()
print(ols_model.summary())

# ====================== IV（2SLS） ======================
print("\n===== IV 2SLS =====")

y = df["stars"]
X = sm.add_constant(df[["log_review"]])   # 控制变量
endog = df["price_level"]                 # 内生变量
iv = df["iv_price"]                       # 工具变量

iv_model = IV2SLS(y, X, endog, iv).fit()

print(iv_model.summary)

# ====================== 内生性检验 ======================
print("\n===== 内生性检验（Wu-Hausman） =====")
print(iv_model.wu_hausman)


===== OLS =====
                            OLS Regression Results                            
Dep. Variable:                  stars   R-squared:                       0.062
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     92.02
Date:                Thu, 02 Apr 2026   Prob (F-statistic):           2.02e-39
Time:                        18:48:46   Log-Likelihood:                -2642.1
No. Observations:                2774   AIC:                             5290.
Df Residuals:                    2771   BIC:                             5308.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           2.8365      0.047

In [9]:
iv_model.wu_hausman()

Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 1.2727
P-value: 0.2594
Distributed: F(1,2770)
WaldTestStatistic, id: 0x14641d46dc0